# Predicting Diabetes with Machine Learning

(Please note that this and the following Colab Notebooks are read-only, so save a copy to your Drive and move it to whichever folder suits you: File-->Save a copy in Drive).

Diabetes is one of the leading causes of death worldwide (8th in the US). One of the main problems is correct diagnosis, as well as prediction and prevention.

For us, it is an opportunity to use real data to learn some basic concepts and techniques:

* How we view/process/clean data
* A first model: Shallow decision trees


We download data from the National Institute of Diabetes and Digestive and Kidney Diseases from here:

https://www.kaggle.com/datasets/mathchi/diabetes-data-set/data

or from the link below.

<img src="https://www.aces.edu/wp-content/uploads/2022/03/FCS-2561-DEEP-Diabetes-Complications-Flyer081621L.jpg" width=250px/>

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```

### The Main Goals of Colab Notebook

The basic idea of **supervised learning**: From examples $X$ and $y$, we want to learn a **rule** to predict the value of $y$ when we only have the values of $X$. In other words, from the characteristics, or **features**, of $X$, we want to learn how to predict $y$.

This Colab notebook will help us understand: How data is organized, what an algorithm and model are, how we train a model or the parameters of a model, and how we use a trained model to make predictions.

Specifically, it will help us:

1. Learn how to load the data
2. Understand how it is organized, and the idea of $X, y$:
  * Each **row** represents a patient. So row 5, for example, contains the characteristics (or "input data" or **features**) that we know about patient number 5.
  * Each **column** of $X$ corresponds to one of the features. So, as we will see, the third column of $X$ corresponds to blood pressure measurements. The eighth column corresponds to age, and so on.
  * $y$ contains a $0$ or a $1$ for each patient -- these are the "answers" we want to learn to predict from $X$.
3. We also need to learn how to visualize the data, and if necessary, how to "clean" it.
4. How to find the best decision tree -- that is, how to use the data to **train** a decision tree.


# Essential Libraries

* sklearn -- contains many machine learning algorithms
* Pandas -- useful for handling data
* Numpy -- essential library for arithmetic and calculations
* Matplotlib -- for data visualization, etc.

In [4]:
# import important python libraries
import pandas as pd # used for data manipulation and analysis.
import numpy as np # used for numerical computations.
import matplotlib.pyplot as plt # used for data visualization.


In [5]:
# sets the seed for NumPy's random number generator to a specific value, in this case, 42
# running code always gets same random numbers -- useful for debugging and verifying results.
np.random.seed(42)

## Downloading the data

* Download the file to your computer from [this link](https://drive.google.com/file/d/1tq0aqY1Bdz3n2qc3IA0lf7o3QWgb6bXX/view?usp=sharing).
* Save it to your Google Drive.

* We need to give Colab access to the Google Drive where the file is stored -- we do this with the code below.

In [6]:
# Since we are not in a Google Colab environment, we cannot use google.colab to mount Google Drive.
# Instead, we will load the data directly from the local file system.

# Load the diabetes dataset from the local file system
file_path = 'diabetes.csv'
df = pd.read_csv(file_path)

# Display the first few rows of the dataframe to ensure it's loaded correctly
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'diabetes.csv'

### Reading the data

Now we need to tell Colab to read the data. In my Google Drive, I have placed it in the folder
```
'/content/drive/MyDrive/Colab Notebooks/YouTube-Data-Sets/diabetes.csv'
```

You need to find the file in your own Google Drive.

The next command reads the data from Google Drive and stores it in a DataFrame in Pandas. In Pandas, a DataFrame is one of the most basic and popular data structures. It is essentially a two-dimensional table (like an Excel spreadsheet) used to store and manipulate data.

We won't go into the details of Pandas here, but you will see enough commands to be able to use it.

In [ ]:
# Replace 'path_to_your_file.csv' with the actual path to your CSV file
#file_path = '/content/drive/MyDrive/Colab Notebooks/path_to_your_file.csv'

file_path = 'diabetes.csv'
diabetes_data = pd.read_csv(file_path) #reads the CSV file located at file_path,
                                       #loads the data into a Pandas DataFrame
                                       #and assigns it to the variable diabetes_data.

### We always look at the data!

The first basic rule of machine learning.

In [ ]:
# Display the first few rows of the dataframe
pd.set_option('display.min_rows', 15)

print(diabetes_data.shape) # displays the number of rows and columns of the array
diabetes_data # prints the actual data

## What have we learned?

We see that we have 768 rows, i.e., data from 768 patients. We also have 8 input data, which are the so-called **Features**:

* Pregnancies
* Glucose
* Blood Pressure
* Skin Thickness
* Insulin
* BMI
* Diabetes Pedigree Function
* Age

 and the last column is the one we want to learn to predict: **Outcome**, which in our example is whether someone is actually diabetic.

## What do we want to do?

We want to use the input data (features) to predict the output data (labels), which in our case is the last column, "Outcome."

# How many have diabetes?
How many of the 768 patients included in our data have diabetes?  

In [ ]:
# count the number of rows that have a '1' as an outcome.
diabetes_data['BloodPressure'].value_counts()

## To begin with: Glucose & BMI

Let us examine only two of the input data (features) and output data (labels): (Glucose = sugar, BMI = body mass index), and the "Outcome." This will allow us to visualize the data using a two-dimensional scatter plot.

The code

```
X = diabetes_data[['Glucose', 'BMI']].values
```
converts the two columns 'Glucose' and 'BMI' of the Dataframe ``diabetes_data`` and stores them as a Numpy Array -- another data structure that allows us to use the ``Numpy`` library.
Similarly, the command
```
y = diabetes_data['Outcome'].values
```
does the same with the 'Outcome' column.

In [ ]:
X = diabetes_data[['Glucose', 'BMI']].values #extract Glucose and BMI columns from the DataFrame and converts them into a NumPy array
y = diabetes_data['Outcome'].values #the same for the outcome column
print(X)
print(y)

### Scatter Plot

We use the ``matplotlib`` library for data visualization. We won't go into detail, but we will use it often in these Notebooks, so you will learn many of the basic uses and commands along the way.

In [ ]:
# Create a scatter plot
plt.figure(figsize=(10, 6)) #This function creates a new figure, which is a container for all the plot elements and sets the width of the figure to 10 inches and the height to 6 inches
for i in range(len(X)): #from 1 to 768 which is the length of the array
    if y[i] == 0: #if outcome is 0
        plt.scatter(X[i, 0], X[i, 1], color='blue', label='Outcome 0' if 'Outcome 0' not in plt.gca().get_legend_handles_labels()[1] else "")
    else: #if outcome is 1
        plt.scatter(X[i, 0], X[i, 1], color='red', label='Outcome 1' if 'Outcome 1' not in plt.gca().get_legend_handles_labels()[1] else "")

# Adding labels and title
plt.xlabel('Glucose')
plt.ylabel('BMI')
plt.title('Scatter Plot of Glucose vs BMI')
plt.legend()
plt.show() # plot current figure to screen


### Strange values!

No one has Glucose = 0 or BMI = 0. Let's remove these values from our data.

In [ ]:
# Filter rows where either Glucose or BMI are zero
filtered_data = diabetes_data[(diabetes_data['Glucose'] != 0) & (diabetes_data['BMI'] != 0)]

# Now extract X and y
X = filtered_data[['Glucose', 'BMI']].values
y = filtered_data['Outcome'].values

### Let's look at the scatter plot again

In [ ]:
# Create a scatter plot
plt.figure(figsize=(10, 6))
for i in range(len(X)):
    if y[i] == 0:
        plt.scatter(X[i, 0], X[i, 1], color='blue', label='Outcome 0' if 'Outcome 0' not in plt.gca().get_legend_handles_labels()[1] else "")
    else:
        plt.scatter(X[i, 0], X[i, 1], color='red', label='Outcome 1' if 'Outcome 1' not in plt.gca().get_legend_handles_labels()[1] else "")

# Adding labels and title
plt.xlabel('Glucose')
plt.ylabel('BMI')
plt.title('Scatter Plot of Glucose vs BMI')
plt.legend()
plt.show()

# We have discussed Decision Trees.

We have seen and discussed the basic definition and use of Decision Trees in the lectures-with-slides.

Let's apply these ideas to this simple example, using the ``sklearn`` library.

### A shallow decision tree

We start with the simplest decision tree -- one that has a depth of one. This corresponds to separating the data based on a threshold in the value of a "feature."

Of course, we will find in an automated/algorithmic way which of the data, and which threshold, achieves the best accuracy in the data.

Our goal is always to understand through simple examples ideas that we can then apply to much more complex problems!

In [ ]:
# import the necessary libraries
from sklearn import tree #import the tree module from the scikit-learn library
from sklearn.tree import DecisionTreeClassifier # used to create a decision tree classifier
from sklearn.metrics import accuracy_score # used to calculate the accuracy of a classification model



## The basic training model

We define the model family.
Then we provide the data and find the best model from this family.

```
model = FamilyOfModels(some parameters)
model.fit(X,y)
```

In our case:

Here we define the family: all decision trees with depth 1. As we have discussed, this family has 4 parameters.
```
decision_stump = DecisionTreeClassifier(max_depth=1)
```
And this is where we train the algorithm: we find the best model from the family we defined above -- that is, we find values for the 4 parameters of the family so that the resulting model (tree) agrees as much as possible with our data: $(X,y)$.
```
decision_stump.fit(X, y)
```



In [ ]:
# Create a decision stump
decision_stump = DecisionTreeClassifier(max_depth=1)

# Fit the model on the training data
decision_stump.fit(X, y) # The fit function is a method to train a machine learning model.
                    # Note that .fit requires training data.
                    # Also referred to as "fitting the model."


# Make predictions
y_pred = decision_stump.predict(X) # generate predictions from a trained machine learning model.

# Compute accuracy
accuracy = accuracy_score(y, y_pred) # compares the true labels with  predicted labels produced by the model and computes the accuracy
                                     # number of correct predictions / total number of predictions
print("Accuracy of the decision stump:", accuracy)

### How well did we do?

We use

```
accuracy_score
```
otherwise we could write a function ourselves (as an exercise) to calculate it.



## What does our tree look like?

What are the values of the four parameters that give the greatest accuracy to the training data?

1. Which feature does it choose for the separation?
2. What threshold does it find to make the separation?
3. What value (label) does it give to the data that exceed the threshold?
4. And what about those that do not exceed it?

In [ ]:
tree.plot_tree(decision_stump, impurity=False, )

### Let's look at it in the graph

In [ ]:
# Define the grid range based on your data
x_min, x_max = X[:, 0].min() - 5, X[:, 0].max() + 5
y_min, y_max = X[:, 1].min() - 5, X[:, 1].max() + 5

# Create a meshgrid
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# Predict the outcome on the meshgrid
Z = decision_stump.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot the decision boundary
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.coolwarm)

# Plot the training points
scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolor='k')
plt.xlabel('Glucose')
plt.ylabel('BMI')
plt.title('Decision Stump for Diabetes Prediction')
plt.legend(handles=scatter.legend_elements()[0], labels=['Non-diabetic', 'Diabetic'])
plt.show()


### Deeper Decision Trees

We will do it again, but with a decision tree depth of 2.
To be precise, **we will do it again together**

We have left some gaps. Try to fill them in, using what we did above, of course.

In [ ]:
# Create a decision tree of depth 2
depth_two_tree = DecisionTreeClassifier(max_depth=2)



# Fit the model on the training data
depth_two_tree.fit(X,y)





# Make predictions
y_pred2 = depth_two_tree.predict(X)



# Compute accuracy
accuracy = accuracy_score(y, y_pred2)
print("Accuracy of the depth two decision tree:", accuracy)

## The separation of space

In [ ]:
# Define the grid range based on your data
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

# Create a meshgrid
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# Predict the outcome on the meshgrid
Z2 = depth_two_tree.predict(np.c_[xx.ravel(), yy.ravel()])
Z2 = Z2.reshape(xx.shape)

# Plot the decision boundary
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z2, alpha=0.8, cmap=plt.cm.coolwarm)

# Plot the training points
scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolor='k')
plt.xlabel('Glucose')
plt.ylabel('BMI')
plt.title('Depth two tree for Diabetes Prediction')
plt.legend(handles=scatter.legend_elements()[0], labels=['Non-diabetic', 'Diabetic'])
plt.show()


# Classification with Logistic Regression and Gradient Boosting

The model we saw above is similar to how we use (select and train) other machine learning algorithms.

We will not go into detail about these specific algorithms in this lecture/notebook. But we will give two more examples:

  A. Logistic Regression -- finds a good linear separation of the data

  B. Gradient Boosting -- uses many shallow decision trees to find a complex classification rule.

**The basics we want to see and understand:**

* Although the two algorithms (logistic regression & gradient boosting) are different from decision trees, the usage pattern is similar:

```
model = FamilyOfModels(some parameters)
model.fit(X.y)
```


In [ ]:
# Fit logistic regression model

# import LogisticRegression
from sklearn.linear_model import LogisticRegression

# declare the family (Logistic Regression)
LR_model = LogisticRegression()

# fit the model to the data
LR_model.fit(X, y)

### How accurate are we?

In [ ]:
# Compute accuracy
y_pred_lr = LR_model.predict(X)
accuracy = accuracy_score(y, y_pred_lr)
print("Accuracy of Logistic Regression:", accuracy)

### We illustrate the separation of space

The code is almost identical to the one we used above.

In [ ]:
# Define the grid range based on your data
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

# Create a meshgrid
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# Predict the outcome on the meshgrid
Z2 = LR_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z2 = Z2.reshape(xx.shape)

# Plot the decision boundary
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z2, alpha=0.8, cmap=plt.cm.coolwarm)

# Plot the training points
scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolor='k')
plt.xlabel('Glucose')
plt.ylabel('BMI')
plt.title('Logistic Regression for Diabetes Prediction')
plt.legend(handles=scatter.legend_elements()[0], labels=['Non-diabetic', 'Diabetic'])
plt.show()


### Gradient Boosting

Uses many shallow decision trees to find a complex classification rule.

In [ ]:
# import xgboost
from xgboost import XGBClassifier

# declare the family (Gradient Boosting Classifier)
GB_model = XGBClassifier()

# fit the model on the training data
GB_model.fit(X, y)

# compute accuracy
y_pred_lr = GB_model.predict(X)
accuracy = accuracy_score(y, y_pred_lr)
print("Accuracy of Gradient Boosting:", accuracy)

### Impressive accuracy!

We will look at this more closely in the next lecture, and we will see that beyond this accuracy in the training data, there is a problem. You can think about it yourself, using the tools we have seen above.

# For the Next Lesson: Overfitting

We saw above that XGBoost achieves very high accuracy on the data it was trained on.

The same happens if, instead of one- or two-depth decision trees, we train
**deeper decision trees**.

In [7]:
# set depth

d = 15 # depth of the tree

# Create a decision tree
deeptree = DecisionTreeClassifier(max_depth=d)

# Fit the model on the training data
deeptree.fit(X, y)
# Make predictions
y_pred = deeptree.predict(X)

# Compute accuracy
accuracy = accuracy_score(y, y_pred)
print("Accuracy of the deep decision tree:", accuracy)

NameError: name 'DecisionTreeClassifier' is not defined

# Diabetes -- Part II -- **Exercise**

You will repeat the logic we followed in the first half of the Colab Notebook to do the following:

1. Using other features -- for example, "Age" & "Blood Pressure":
  * Select these two features, as we did initially for BMI and Glucose.
  * Use model.fit to find the best depth 1 and 2 trees on these two features.
  * Visualize the resulting space partition.
2. Compare which is better (in terms of accuracy) — the results with these two features, or with the first two we selected.
3. Then, look at more than 2 features -- now we can no longer visualize them since we need more than the 2 dimensions of the image, but we can use the same algorithms -- depth 1 and depth 2 decision trees, to see if we do better using all features at the same time.

### Συμπληρώστε τα κενά και τα ΧΧΧΧΧ

In [ ]:
# θα ξεδιαλέξουμε τα χαρακτηριστικά 'Age' και 'Blood Pressure' -- χρησιμοποιούμε το filtered_data
X = XXXXX
# τα outcomes
y = XXXXX

# Create a scatter plot
plt.figure(figsize=(10, 6))
for i in range(len(X)):
    if y[i] == 0:
        plt.scatter(X[i, 0], X[i, 1], color='blue', label='Outcome 0' if 'Outcome 0' not in plt.gca().get_legend_handles_labels()[1] else "")
    else:
        plt.scatter(X[i, 0], X[i, 1], color='red', label='Outcome 1' if 'Outcome 1' not in plt.gca().get_legend_handles_labels()[1] else "")

# Adding labels and title
plt.xlabel('XXXXX')
plt.ylabel('XXXXX')
plt.title('XXXXX')
plt.legend()
plt.show()

### Do we have any unusual values?

If there are any unusual values as before, remove them, thus creating a new dataframe as we did initially.


In [ ]:
# remove the rows (patients)
# with values that do not make sense
# use code similar to the one
# we did together.

XXXXX

# redefine X and y
X = XXXXX
# the outcomes
y = XXXXX

### Decision trees

Once again, we can see how well a shallow decision tree performs.

* Accuracy?
* Better, worse, or the same as before?

In [ ]:
# train a shallow decision tree on the two
# features 'Age' and 'Blood Pressure'
# you will use similar code to the one
# we did together.


### What does the division of space look like?

In [ ]:
XXXXX # Please note that you can copy the code from above.

## Using All Data

We can now try to use all the data. Although we cannot see the data on the screen, to find the best decision tree (depth d) we can follow exactly the same logic, using the same commands.

In [8]:
# Data
X_full = morefiltered_data.values
y = morefiltered_data['Outcome'].values


# Create a decision tree
model = DecisionTreeClassifier(max_depth=1)


# Fit the model on the training data
model.fit(X_full,y)


# Make predictions
y_pred2 = model.predict(X_full)




# Compute accuracy
accuracy = accuracy_score(y, y_pred2)
print("Accuracy of the decision tree:", accuracy)

NameError: name 'morefiltered_data' is not defined

## Puzzle

**What (fatal) mistake did we make?
Why does it give us 100% accuracy?**


Answer:

Because $X_{\rm full}$ also contains the **Outcome** column!
It is very easy to overlook such errors. We must always use our judgment when we see a result: "Why is it so good? Does it make sense?" or also "Why didn't it work as I expected?"... Artificial intelligence is still... artificial... it hasn't replaced us yet!


You can use the command
```
tree.plot_tree(model, impurity=False)
```
to see which tree achieved 100% accuracy -- you will see how it made decisions based on the last column!

In [ ]:
tree.plot_tree(model, impurity=False)

In [ ]:
# Let's go again!
X_full = morefiltered_data.loc[:, filtered_data.columns != 'Outcome'].values

# Create a decision tree
tree_d2 = DecisionTreeClassifier(max_depth=2)

# Fit the model on the training data
tree_d2.fit(X_full, y)
# Make predictions
y_pred2 = tree_d2.predict(X_full)

# Compute accuracy
accuracy = accuracy_score(y, y_pred2)
print("Accuracy of the decision stump:", accuracy)

# The above input data did not help us in the end

We did as well as we could using only BMI & Glucose.

In the next lesson, we will return to this example to better understand what is happening, what else we could do, and, most importantly, why the high accuracy achieved by XGBoost and deep decision trees does not ultimately offer the magic solution we are looking for.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3421b538-369d-47a6-a492-819e6e3738bb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>